<a href="https://colab.research.google.com/github/sheicksen/CISC483-EngageTactics/blob/JiaQi-RNN-predict_YouTube_engagement/RNN_on_YouTube.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import kagglehub
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import transformers

To load the tokenizer from the saved JSON file, you can use `tf.keras.preprocessing.text.tokenizer_from_json()`.

In [19]:
import json
from tensorflow.keras.preprocessing.text import tokenizer_from_json

# Load the tokenizer from the JSON file
with open('tokenizer.json', 'r', encoding='utf-8') as f:
    loaded_tokenizer_json = json.load(f)
    loaded_tokenizer = tokenizer_from_json(loaded_tokenizer_json)

print("Tokenizer loaded successfully.")
# You can verify it by checking its word_index
print(f"Loaded tokenizer vocabulary size: {len(loaded_tokenizer.word_index)}")

Tokenizer loaded successfully.
Loaded tokenizer vocabulary size: 66620


Now that the tokenizer is loaded, you can use it to encode text. Here's a quick example:

In [21]:
sample_text = "This is a sample sentence to tokenize."
encoded_input = loaded_tokenizer.texts_to_sequences([sample_text]) # texts_to_sequences expects a list of texts

print("Original text:", sample_text)
print("Encoded input:", encoded_input)

Original text: This is a sample sentence to tokenize.
Encoded input: [[24, 8, 6, 7692, 3648, 3, 1]]


In [8]:
## Data is stored in the cache for ease of access / disposal
yt_path = kagglehub.dataset_download("bsthere/youtube-trending-videos-stats-2026")

print("Path to YouTube dataset files:", yt_path)

Using Colab cache for faster access to the 'youtube-trending-videos-stats-2026' dataset.
Path to YouTube dataset files: /kaggle/input/youtube-trending-videos-stats-2026


In [10]:
## Import US YouTube Data
yt_df = pd.read_csv(yt_path + "/US_Trending.csv")

print(yt_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16400 entries, 0 to 16399
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   video_id       16400 non-null  object
 1   trending_date  16400 non-null  object
 2   title          16400 non-null  object
 3   channel_title  16399 non-null  object
 4   views          16400 non-null  int64 
 5   likes          16400 non-null  int64 
 6   dislikes       16400 non-null  int64 
 7   publish_time   16400 non-null  object
 8   category_id    16400 non-null  int64 
 9   tags           16400 non-null  object
 10  comments       16400 non-null  int64 
 11  channel_id     16400 non-null  object
 12  description    13580 non-null  object
dtypes: int64(5), object(8)
memory usage: 1.6+ MB
None


In [28]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# what the model will classify based on (title & description)
X = yt_df['title'] + " [SEP] " + yt_df['description']

MAX_SEQUENCE_LENGTH = 150 # longest the text can be

# Fill NaN values with empty strings before tokenizing
X_processed = X.fillna('')
X_tokenized = loaded_tokenizer.texts_to_sequences(X_processed)

# Pad sequences to ensure uniform length
padded_X_tokenized = pad_sequences(X_tokenized, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
print("Example after tokenizing and padding:", padded_X_tokenized[0])

Example after tokenizing and padding: [   1    1   16 1008 6036    1  792  406    1 1302   15    1    1 5675
 8525 5974 5675 4172    1    1    1 8031 8525 5974 8031 4172    1    1
    1    1    2    1   54   69 1538  261 8525    1    1    3    1    1
    1 1551    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0]
